# AI Word Correction — Visual Step-by-Step
Each cell is one step of the pipeline. Run them top-to-bottom.

In [1]:
# ── STEP 0: CONFIG ─────────────────────────────────────────────────────────
import sys, os, io, json, base64, pathlib

# Add project root to path so word_constructor is importable
PROJECT_ROOT = str(pathlib.Path("__file__").resolve().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Load .env (picks up OPENAI_API_KEY, model, base_url, etc.)
env_path = pathlib.Path(PROJECT_ROOT) / ".env"
if env_path.exists():
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, _, v = line.partition("=")
            os.environ.setdefault(k.strip(), v.strip())
    print(f"Loaded .env from {env_path}")
else:
    print(f"No .env at {env_path} — set OPENAI_API_KEY manually if needed")

# ── Load from example_replacement.json ────────────────────────────────────
EXAMPLE_JSON = pathlib.Path(PROJECT_ROOT) / "example_replacement.json"
with EXAMPLE_JSON.open(encoding="utf-8") as f:
    example = json.load(f)

# Decode embedded docx bytes
DOC_BYTES = io.BytesIO(base64.b64decode(example["content_base64"]))
DOC_FILENAME = example.get("filename", "template.docx")

# Placeholders: keep only string values (skip МассивШапки lists, meta keys)
_META_KEYS = {"ИспользоватьAI", "PromtAI"}
PARAMS: dict[str, str] = {
    k: str(v)
    for k, v in example["placeholders"].items()
    if isinstance(v, str) and k not in _META_KEYS
}

PROMPT_AI: str = example.get("PromtAI", "")
USE_AI: bool = str(example.get("UseAI", "true")).lower() == "true"

print(f"\nFile    : {DOC_FILENAME}")
print(f"UseAI   : {USE_AI}")
print(f"PromtAI : {PROMPT_AI[:80]!r}{'…' if len(PROMPT_AI) > 80 else ''}")
print(f"\nParams ({len(PARAMS)} placeholders):")
for k, v in list(PARAMS.items())[:15]:
    print(f"  [{k}] = {str(v)[:70]!r}")
if len(PARAMS) > 15:
    print(f"  … and {len(PARAMS) - 15} more")

Loaded .env from C:\Users\User\Desktop\word-portal\.env

File    : template.docx
UseAI   : True
PromtAI : 'Дополнительные правила для этого документа (приоритет выше общих правил, если ес'…

Params (9 placeholders):
  [Ссылка] = 'Прием на работу 0000-000233 от 15.12.2025'
  [СсылкаДолжностьРуководителяНаименование] = 'hr бизнес-партнер'
  [СсылкаДатаПриема] = '15.12.2025'
  [РеквизитыСотрудникПодразделениеНаименование] = 'компас воад'
  [СсылкаСотрудникФизическоеЛицоФИО] = 'Садық Ермек Жәнібекұлы'
  [РеквизитыОтветственныйФизическоеЛицоФИО] = 'Сатымбаева Салтанат Мусаевна'
  [РеквизитыРуководительФИО] = 'Есжанова Зарина Серикалиевна'
  [РеквизитыДолжностьРуководителяНаименование] = 'hr бизнес-партнер'
  [РеквизитыСотрудникДолжностьНаименование] = 'кассир-повар'


In [2]:
# ── STEP 1: LOAD DOCUMENT ──────────────────────────────────────────────────
from docx import Document

doc = Document(DOC_BYTES)

para_count = len(doc.paragraphs)
table_count = len(doc.tables)
section_count = len(doc.sections)

print(f"Document loaded: {DOC_FILENAME}")
print(f"  Paragraphs : {para_count}")
print(f"  Tables     : {table_count}")
print(f"  Sections   : {section_count}")

print("\nFirst 5 non-empty paragraphs:")
shown = 0
for p in doc.paragraphs:
    text = p.text.strip()
    if text:
        print(f"  {text[:120]}")
        shown += 1
        if shown >= 5:
            break

Document loaded: template.docx
  Paragraphs : 32
  Tables     : 2
  Sections   : 1

First 5 non-empty paragraphs:
  О приеме на работу
  В соответствии со статьей 34 Трудового кодекса Республики Казахстан, ПРИКАЗЫВАЮ:
  Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение
  Основание: трудовой договор № [НомерДоговора] от [ДатаНачалаДоговора]года, заявление от [СсылкаСотрудникФизическоеЛицоФИ
  С приказом ознакомлен (а):


In [3]:
# ── STEP 2: WALK DOCUMENT → TextUnits ─────────────────────────────────────
from word_constructor.ai_correction import walk_document

text_units = walk_document(doc)

print(f"Total TextUnits: {len(text_units)}")
print()

type_counts: dict[str, int] = {}
for u in text_units:
    type_counts[u.source_type] = type_counts.get(u.source_type, 0) + 1
print("By source_type:")
for t, c in type_counts.items():
    print(f"  {t:25s} {c}")

print()
print("── First 10 non-empty units ──")
shown = 0
for u in text_units:
    if u.text.strip():
        ai_flag = " [AI-EXCLUDED]" if u.ai_excluded else ""
        print(f"  [{u.source_type}] {u.source_path}{ai_flag}")
        print(f"    {u.text[:100]!r}")
        shown += 1
        if shown >= 10:
            break

Total TextUnits: 46

By source_type:
  body_paragraph            32
  table_cell                14

── First 10 non-empty units ──
  [table_cell] table[0].row[0].cell[0]
    'БҰЙРЫҚ'
  [table_cell] table[0].row[0].cell[3]
    'ПРИКАЗ'
  [table_cell] table[0].row[1].cell[0]
    '<ДатаДокумента> года'
  [table_cell] table[0].row[1].cell[1]
    '<ДатаДокумента> года'
  [table_cell] table[0].row[1].cell[2]
    '№ <РегНомерДокумента>'
  [table_cell] table[0].row[1].cell[3]
    '№ <РегНомерДокумента>'
  [table_cell] table[0].row[2].cell[0]
    'Алматы қаласы'
  [table_cell] table[0].row[2].cell[1]
    'Алматы қаласы'
  [table_cell] table[0].row[2].cell[2]
    'город Алматы'
  [table_cell] table[0].row[2].cell[3]
    'город Алматы'


In [4]:
# ── STEP 3: FIND OCCURRENCES ──────────────────────────────────────────────
from word_constructor.ai_correction import find_occurrences

occurrences = find_occurrences(text_units, PARAMS)

print(f"Found {len(occurrences)} occurrence(s)")
print()

for occ in occurrences:
    excluded = " ⛔ AI-EXCLUDED" if occ.ai_excluded else ""
    print(f"  id             : {occ.id}")
    print(f"  placeholder    : [{occ.placeholder}]")
    print(f"  occurrence_idx : {occ.occurrence_index}")
    print(f"  original_value : {occ.original_value!r}")
    print(f"  source_type    : {occ.source_type}")
    print(f"  source_path    : {occ.source_path}{excluded}")
    print(f"  context_text   : {occ.context_text[:120]!r}")
    print()

Found 7 occurrence(s)

  id             : СсылкаСотрудникФизическоеЛицоФИО#0
  placeholder    : [СсылкаСотрудникФизическоеЛицоФИО]
  occurrence_idx : 0
  original_value : 'Садық Ермек Жәнібекұлы'
  source_type    : body_paragraph
  source_path    : paragraph[9]
  context_text   : 'Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение'

  id             : РеквизитыСотрудникДолжностьНаименование#0
  placeholder    : [РеквизитыСотрудникДолжностьНаименование]
  occurrence_idx : 0
  original_value : 'кассир-повар'
  source_type    : body_paragraph
  source_path    : paragraph[9]
  context_text   : 'Принять [СсылкаСотрудникФизическоеЛицоФИО] на [РеквизитыСотрудникДолжностьНаименование] [РеквизитыСотрудникПодразделение'

  id             : РеквизитыСотрудникПодразделениеНаименование#0
  placeholder    : [РеквизитыСотрудникПодразделениеНаименование]
  occurrence_idx : 0
  original_value : 'компас воад'
  source_type    : body_p

In [5]:
# ── STEP 3b: REDUNDANCY / DUPLICATE DETECTION ─────────────────────────────
# When [Должность] already contains the full [Подразделение] value,
# the Подразделение occurrence is marked redundant → will be blanked out.
#
# Detection runs automatically inside find_occurrences / extract_placeholder_occurrences.
# This cell shows what was detected, plus a concrete demo with overlapping values.

print("── Redundancy detection on actual PARAMS ──")
redundant = [o for o in occurrences if o.redundant_in]
if redundant:
    for o in redundant:
        adj_id = o.redundant_in
        adj = next((x for x in occurrences if x.id == adj_id), None)
        print(f"  ⚠  [{o.placeholder}#{o.occurrence_index}]")
        print(f"       value         : {o.original_value!r}")
        print(f"       embedded in   : [{adj.placeholder}#{adj.occurrence_index}] = {adj.original_value!r}" if adj else f"       embedded in   : {adj_id}")
else:
    print("  (no redundancies in current PARAMS — values don't overlap)")

print()
print("── Demo: overlapping values that WOULD trigger dedup ──")
from word_constructor.ai_correction import find_occurrences as _find

# Simulate a document paragraph: "[Должность] [Подразделение] АО «Halyk Finance»"
from word_constructor.ai_correction.types import TextUnit as _TU
demo_units = [
    _TU(
        source_type="body_paragraph",
        source_path="paragraph[demo]",
        text="Принять сотрудника на [ДолжностьДемо] [ПодразделениеДемо] АО «Halyk Finance»",
    )
]
demo_params = {
    "ДолжностьДемо":    "Главный менеджер департамента кадровой политики",
    "ПодразделениеДемо": "департамент кадровой политики",
}
demo_occs = _find(demo_units, demo_params)

for o in demo_occs:
    flag = f"  ⚠  REDUNDANT (embedded in [{o.redundant_in}])" if o.redundant_in else "  ✅ kept"
    print(f"\n  [{o.placeholder}#{o.occurrence_index}]{flag}")
    print(f"    value : {o.original_value!r}")

── Redundancy detection on actual PARAMS ──
  (no redundancies in current PARAMS — values don't overlap)

── Demo: overlapping values that WOULD trigger dedup ──

  [ДолжностьДемо#0]  ✅ kept
    value : 'Главный менеджер департамента кадровой политики'

  [ПодразделениеДемо#0]  ⚠  REDUNDANT (embedded in [ДолжностьДемо#0])
    value : 'департамент кадровой политики'


In [6]:
# ── STEP 4: SANITY CHECK ──────────────────────────────────────────────────
from word_constructor.ai_correction import sanity_check

ok, messages = sanity_check(text_units, occurrences)

status = "✅ PASSED" if ok else "❌ FAILED"
print(f"Sanity check: {status}")
if messages:
    for msg in messages:
        print(f"  {msg}")
else:
    print(f"  raw regex matches == occurrences ({len(occurrences)}) — counts match")

Sanity check: ❌ FAILED
  raw placeholder regex matches=9, occurrences=7


In [7]:
# ── STEP 5: EXTRACT FULL DOCUMENT TEXT (headers + body + footers) ──────────
# This single block is now the entire context sent to AI.
# AI reads the whole document and determines the correct grammatical form
# of each placeholder from the surrounding prose — no per-placeholder snippets needed.

from word_constructor.ai_correction import document_full_text

full_doc_text = document_full_text(doc)

char_count = len(full_doc_text)
line_count = full_doc_text.count("\n") + 1
section_labels = [ln for ln in full_doc_text.splitlines() if ln.startswith("[")]

print(f"Full document text: {char_count} chars, {line_count} lines")
print(f"Sections / text-units included: {len(section_labels)}")
print()
print("── Section labels (headers first, then body) ──")
for label in section_labels:
    print(f"  {label}")
print()
print("── Full text preview (first 1 500 chars) ──")
print(full_doc_text[:1500])

Full document text: 1491 chars, 56 lines
Sections / text-units included: 21

── Section labels (headers first, then body) ──
  [table_cell table[0].row[0].cell[0]]
  [table_cell table[0].row[0].cell[3]]
  [table_cell table[0].row[1].cell[0]]
  [table_cell table[0].row[1].cell[1]]
  [table_cell table[0].row[1].cell[2]]
  [table_cell table[0].row[1].cell[3]]
  [table_cell table[0].row[2].cell[0]]
  [table_cell table[0].row[2].cell[1]]
  [table_cell table[0].row[2].cell[2]]
  [table_cell table[0].row[2].cell[3]]
  [body_paragraph paragraph[5]]
  [body_paragraph paragraph[7]]
  [body_paragraph paragraph[9]]
  [body_paragraph paragraph[11]]
  [table_cell table[1].row[0].cell[0]]
  [РеквизитыДолжностьРуководителяНаименование]
  [table_cell table[1].row[0].cell[1]]
  [РеквизитыРуководительФИО]
  [body_paragraph paragraph[26]]
  [body_paragraph paragraph[27]]
  [body_paragraph paragraph[31]]

── Full text preview (first 1 500 chars) ──
[table_cell table[0].row[0].cell[0]]
БҰЙРЫҚ

[table_cell t

In [8]:
# ── STEP 6: BUILD OPENAI REQUEST PAYLOAD ──────────────────────────────────
# Now uses the full document text (with headers/footers) as the single context.
# The per-placeholder context snippets are no longer included in the payload.

from word_constructor.ai_correction.extraction import extract_placeholder_occurrences
from word_constructor.ai_correction.openai_client import openai_placeholder_payload
from word_constructor.ai_correction.rules import load_rules_config

rules = load_rules_config()
raw_occurrences = extract_placeholder_occurrences(doc, PARAMS, rules)

payload = openai_placeholder_payload(
    slot_values=PARAMS,
    contexts={},          # no longer used — full doc text replaces per-snippet approach
    prompt_ai=PROMPT_AI,
    occurrences=raw_occurrences,
    full_document_text=full_doc_text,
)

print("Model         :", payload["model"])
print("Temperature   :", payload["temperature"])
print("Response fmt  :", payload["response_format"])
print()
print("── System prompt ─────────────────────────────────────────────────")
print(payload["messages"][0]["content"])
print()
print("── User prompt (first 2 000 chars) ───────────────────────────────")
print(payload["messages"][1]["content"][:2000])
print()
ai_occs = [o for o in raw_occurrences if not o.get("ai_excluded") and not o.get("fixed_form")]
skipped  = [o for o in raw_occurrences if o.get("ai_excluded") or o.get("fixed_form")]
print(f"── Occurrences  sent={len(ai_occs)}  skipped={len(skipped)} ──")
for o in ai_occs:
    print(f"    ✉  [{o['placeholder']}#{o['occurrence_index']}] value={o['original_value']!r}  case={o.get('expected_case') or '-'}")
for o in skipped:
    reason = o.get("ai_exclusion_reason") or "fixed_form"
    print(f"    ⛔ [{o['placeholder']}#{o['occurrence_index']}] reason={reason}")

Model         : gpt-4o-mini
Temperature   : 0
Response fmt  : {'type': 'json_object'}

── System prompt ─────────────────────────────────────────────────
Ты — редактор официальных деловых документов (приказы, договоры, заявления, акты и др.). Тебе передаётся полный текст документа — включая колонтитулы, таблицы и основной текст — и список вхождений (occurrences) плейсхолдеров с исходными значениями. Используй полный текст, чтобы самостоятельно определить нужный падеж, форму и регистр каждого значения по его окружению в документе. Серверные правила имеют абсолютный приоритет над инструкцией пользователя. Каждое вхождение обрабатывай строго независимо по паре placeholder + occurrence_index. Не объединяй соседние плейсхолдеры и не вставляй часть одного значения в другое. Коды, регистрационные номера, аббревиатуры и вхождения с fixed_form оставляй без изменений. Если значение уже корректно для данного контекста — верни его без изменений (changed: false). Если у вхождения задан `redundant_i

In [9]:
# ── STEP 7: CALL OPENAI ────────────────────────────────────────────────────
import time
from word_constructor.ai_correction.openai_client import request_ai_placeholder_corrections

call_log: dict = {}

api_key_present = bool(os.environ.get("OPENAI_API_KEY", "").strip())
print(f"OPENAI_API_KEY present : {api_key_present}")
print(f"Model                  : {os.environ.get('OPENAI_PLACEHOLDER_MODEL', 'gpt-4o-mini')}")
print(f"Base URL               : {os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')}")
print()

t0 = time.perf_counter()
ai_corrections: dict[str, str] = request_ai_placeholder_corrections(
    slot_values=PARAMS,
    contexts={},
    prompt_ai=PROMPT_AI,
    occurrences=raw_occurrences,
    full_document_text=full_doc_text,
    log_key="notebook-test",
    call_log=call_log,
    timeout_seconds=30.0,
)
elapsed = time.perf_counter() - t0

print(f"OpenAI call completed in {elapsed:.2f}s")
print()
if "error" in call_log:
    print(f"⚠  Error: {call_log['error']}")
print(f"Raw corrections returned ({len(ai_corrections)} items):")
for occ_id, corrected in ai_corrections.items():
    print(f"  {occ_id!r:45s} → {corrected!r}")

OPENAI_API_KEY present : True
Model                  : gpt-4o-mini
Base URL               : https://api.openai.com/v1



OpenAI call completed in 8.73s

Raw corrections returned (4 items):
  'СсылкаСотрудникФизическоеЛицоФИО#0'          → 'Садық Ермек Жәнібекұлы'
  'РеквизитыСотрудникДолжностьНаименование#0'   → 'кассира-повара'
  'СсылкаДатаПриема#0'                          → '15 декабря 2025 года'
  'СсылкаСотрудникФизическоеЛицоФИО#1'          → 'Садық Ермек Жәнібекұлы'


In [10]:
# ── STEP 8: INSPECT RAW OPENAI RESPONSE ───────────────────────────────────
if call_log.get("response"):
    resp = call_log["response"]
    print(f"HTTP status : {resp.get('status')}")
    print(f"Bytes       : {resp.get('bytes')}")
    print()
    print("── Response body (JSON) ──")
    print(json.dumps(resp.get("body", {}), ensure_ascii=False, indent=2)[:3000])
else:
    print("No response recorded in call_log (API key missing or error before request).")

HTTP status : 200
Bytes       : 2020

── Response body (JSON) ──
{
  "id": "chatcmpl-Dtpu6HGFCXxtxas4eNvBNitmRjvW9",
  "object": "chat.completion",
  "created": 1782199930,
  "model": "gpt-4o-mini-2024-07-18",
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "{\n  \"occurrences\": [\n    {\n      \"placeholder\": \"СсылкаСотрудникФизическоеЛицоФИО\",\n      \"occurrence_index\": 0,\n      \"original_value\": \"Садық Ермек Жәнібекұлы\",\n      \"corrected_value\": \"Садық Ермек Жәнібекұлы\",\n      \"changed\": false\n    },\n    {\n      \"placeholder\": \"РеквизитыСотрудникДолжностьНаименование\",\n      \"occurrence_index\": 0,\n      \"original_value\": \"кассир-повар\",\n      \"corrected_value\": \"кассира-повара\",\n      \"changed\": true\n    },\n    {\n      \"placeholder\": \"СсылкаДатаПриема\",\n      \"occurrence_index\": 0,\n      \"original_value\": \"15.12.2025\",\n      \"corrected_value\": \"15 декабря 2025 года\

In [11]:
# ── STEP 9: DIFF — ORIGINAL vs AI CORRECTED ───────────────────────────────
# Map correction_id → corrected value, then join with occurrences for display.

id_to_occ = {o["id"]: o for o in raw_occurrences}

print(f"{'PLACEHOLDER':<30} {'OCC':>4}  {'ORIGINAL VALUE':<35} {'AI CORRECTED VALUE':<35} CHANGED")
print("-" * 120)

for occ in raw_occurrences:
    occ_id = occ["id"]
    original = occ["original_value"]
    if occ.get("ai_excluded"):
        corrected = original
        label = "⛔ excluded"
    elif occ.get("fixed_form"):
        corrected = original
        label = "🔒 fixed_form"
    else:
        corrected = ai_corrections.get(occ_id, original)
        label = "✅ changed" if corrected != original else "  same"
    print(f"  [{occ['placeholder']}#{occ['occurrence_index']}]")
    print(f"    Original  : {original!r}")
    print(f"    Corrected : {corrected!r}  {label}")
    print()

PLACEHOLDER                     OCC  ORIGINAL VALUE                      AI CORRECTED VALUE                  CHANGED
------------------------------------------------------------------------------------------------------------------------
  [СсылкаСотрудникФизическоеЛицоФИО#0]
    Original  : 'Садық Ермек Жәнібекұлы'
    Corrected : 'Садық Ермек Жәнібекұлы'    same

  [РеквизитыСотрудникДолжностьНаименование#0]
    Original  : 'кассир-повар'
    Corrected : 'кассира-повара'  ✅ changed

  [РеквизитыСотрудникПодразделениеНаименование#0]
    Original  : 'компас воад'
    Corrected : 'компас воад'  🔒 fixed_form

  [СсылкаДатаПриема#0]
    Original  : '15.12.2025'
    Corrected : '15 декабря 2025 года'  ✅ changed

  [СсылкаСотрудникФизическоеЛицоФИО#1]
    Original  : 'Садық Ермек Жәнібекұлы'
    Corrected : 'Садық Ермек Жәнібекұлы'    same

  [РеквизитыДолжностьРуководителяНаименование#0]
    Original  : 'hr бизнес-партнер'
    Corrected : 'hr бизнес-партнер'  ⛔ excluded

  [РеквизитыРуково

In [12]:
# ── STEP 10: FULL PIPELINE (all steps in one call) ─────────────────────────
# Uses pipeline.correct_slot_values which internally does everything above.
# decline_func is only needed for grammatical declension after AI.

from word_constructor.ai_correction.pipeline import correct_slot_values

def identity_decline(value: str, case: str) -> str:
    """Fallback: return value unchanged (no morphology library loaded here)."""
    return value

pipeline_log: dict = {}
result = correct_slot_values(
    doc=doc,
    slot_values=PARAMS,
    prompt_ai=PROMPT_AI,
    decline_func=identity_decline,
    call_log=pipeline_log,
)

print("Pipeline result")
print("  ai_skipped_reason :", result.ai_skipped_reason or "(none — AI ran)")
print()
print("Final slot values (original → pipeline output):")
print(f"  {'PLACEHOLDER':<30} {'ORIGINAL':<35} {'FINAL':<35}")
print("  " + "-" * 100)
for key, original in PARAMS.items():
    # occurrence_values keyed by (key, occurrence_number)
    finals = {occ_num: v for (k, occ_num), v in result.occurrence_values.items() if k == key}
    if finals:
        for occ_num, final_val in sorted(finals.items()):
            changed = " ✅" if final_val != original else ""
            print(f"  [{key}] occ#{occ_num:<3} {original:<35} {final_val:<35}{changed}")
    else:
        print(f"  [{key}]          {original:<35} (no AI correction)")

print()
sanity_info = result.sanity_check
print("Sanity check summary:")
print(f"  full_text_raw_match_count : {sanity_info.get('full_text_raw_match_count')}")
print(f"  raw_match_count           : {sanity_info.get('raw_match_count')}")
print(f"  occurrence_count          : {sanity_info.get('occurrence_count')}")

Pipeline result
  ai_skipped_reason : (none — AI ran)

Final slot values (original → pipeline output):
  PLACEHOLDER                    ORIGINAL                            FINAL                              
  ----------------------------------------------------------------------------------------------------
  [Ссылка]          Прием на работу 0000-000233 от 15.12.2025 (no AI correction)
  [СсылкаДолжностьРуководителяНаименование]          hr бизнес-партнер                   (no AI correction)
  [СсылкаДатаПриема] occ#1   15.12.2025                          15 декабря 2025 года                ✅
  [РеквизитыСотрудникПодразделениеНаименование] occ#1   компас воад                         компас воад                        
  [СсылкаСотрудникФизическоеЛицоФИО] occ#1   Садық Ермек Жәнібекұлы              Садық Ермек Жәнібекұлы             
  [СсылкаСотрудникФизическоеЛицоФИО] occ#2   Садық Ермек Жәнібекұлы              Садық Ермек Жәнібекұлы             
  [РеквизитыОтветственныйФизическое

In [13]:
# ── STEP 11: COMPREHENSIVE TYPE DEMO ──────────────────────
# Simulated document covering: dates, days-in-words, bracket format,
# name declension, and title/department deduplication.
# Runs the FULL pipeline on this mock document in-memory.

import sys, io
sys.stdout.reconfigure(encoding="utf-8") if hasattr(sys.stdout, "reconfigure") else None

from docx import Document as _Doc
from word_constructor.ai_correction.pipeline import correct_slot_values
from word_constructor.ai_correction.deterministic import (
    format_ru_date_full, format_days_bracket, format_number_words_ru,
)

# ── Build a mock in-memory .docx ──────────────────────────────────────────
_buf = io.BytesIO()
_d = _Doc()
_d.add_paragraph(
    "ДОГОВОР № [НомерДоговора] от [ДатаДоговора] года"
)
_d.add_paragraph(
    "Настоящий договор заключён сроком на [КолДней] ([КолДнейПрописью]) "
    "календарных дней, с [ДатаНачала] года по [ДатаОкончания] года."
)
_d.add_paragraph(
    "В течение [СрокДней] рабочих дней сторона обязана предоставить документы."
)
_d.add_paragraph(
    "Принять [ФИОСотрудника] на должность [ДолжностьСотрудника] "
    "[ПодразделениеСотрудника]."
)
_d.add_paragraph(
    "Основание: заявление от [ФИОСотрудника]."
)
_d.save(_buf)
_buf.seek(0)
mock_doc = _Doc(_buf)

DEMO_PARAMS = {
    "НомерДоговора":         "ТД-2025-0042",
    "ДатаДоговора":          "15.12.2025",
    "КолДней":               "30",
    "КолДнейПрописью":       "30",
    "ДатаНачала":            "15.12.2025",
    "ДатаОкончания":         "14.01.2026",
    "СрокДней":              "5",
    "ФИОСотрудника":         "иванов иван иванович",
    "ДолжностьСотрудника":   "главный бухгалтер финансового департамента",
    "ПодразделениеСотрудника": "финансовый департамент",
}

def _decline(value: str, case: str) -> str:
    return value  # identity — morphology not loaded in notebook

demo_log: dict = {}
demo_result = correct_slot_values(
    doc=mock_doc,
    slot_values=DEMO_PARAMS,
    prompt_ai="",
    decline_func=_decline,
    call_log=demo_log,
)

print("── Mock document paragraphs ──────────────────────────────────────")
for p in mock_doc.paragraphs:
    if p.text.strip():
        print(f"  {p.text}")

print()
print("── Corrections applied ───────────────────────────────────────────")
print(f"  {'PLACEHOLDER':<35} {'ORIGINAL':<40} {'FINAL'}")
print("  " + "─" * 110)
for key, original in DEMO_PARAMS.items():
    finals = {occ: v for (k, occ), v in demo_result.occurrence_values.items() if k == key}
    if finals:
        for occ_num, final in sorted(finals.items()):
            changed = " ✅" if final != original else ""
            blanked = " 🗑  (deduped)" if final == "" else ""
            print(f"  [{key}] occ#{occ_num}")
            print(f"    orig  : {original!r}")
            print(f"    final : {final!r}{changed}{blanked}")
    else:
        print(f"  [{key}]  →  (no correction applied)")
    print()

print()
print("── Deterministic helper functions ────────────────────────────────")
print(f"  format_ru_date_full('15.12.2025')          → {format_ru_date_full('15.12.2025')!r}")
print(f"  format_days_bracket('30')                  → {format_days_bracket('30')!r}")
print(f"  format_days_bracket('5')                   → {format_days_bracket('5')!r}")
print(f"  format_number_words_ru('30', 'nomn')       → {format_number_words_ru('30', 'nomn')!r}")
print(f"  format_number_words_ru('5',  'gent')       → {format_number_words_ru('5',  'gent')!r}")
print(f"  format_number_words_ru('14', 'datv')       → {format_number_words_ru('14', 'datv')!r}")


AI placeholder correction failed; deterministic safeguards only: use_ai_log_key=None error=The read operation timed out
Traceback (most recent call last):
  File "C:\Users\User\Desktop\word-portal\word_constructor\ai_correction\pipeline.py", line 100, in correct_slot_values
    corrections = request_ai_placeholder_corrections(
        ai_values,
    ...<6 lines>...
        timeout_seconds,
    )
  File "C:\Users\User\Desktop\word-portal\word_constructor\ai_correction\openai_client.py", line 139, in request_ai_placeholder_corrections
    with urlopen(req, timeout=timeout_seconds) as resp:
         ~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 189, in urlopen
    return opener.open(url, data, timeout)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\User\AppData\Local\Programs\Python\Python313\Lib\urllib\request.py", line 489, in open
    response = self._open(req, data)
  File "C:\Users\User\Ap

── Mock document paragraphs ──────────────────────────────────────
  ДОГОВОР № [НомерДоговора] от [ДатаДоговора] года
  Настоящий договор заключён сроком на [КолДней] ([КолДнейПрописью]) календарных дней, с [ДатаНачала] года по [ДатаОкончания] года.
  В течение [СрокДней] рабочих дней сторона обязана предоставить документы.
  Принять [ФИОСотрудника] на должность [ДолжностьСотрудника] [ПодразделениеСотрудника].
  Основание: заявление от [ФИОСотрудника].

── Corrections applied ───────────────────────────────────────────
  PLACEHOLDER                         ORIGINAL                                 FINAL
  ──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  [НомерДоговора] occ#1
    orig  : 'ТД-2025-0042'
    final : 'ТД-2025-0042'

  [ДатаДоговора] occ#1
    orig  : '15.12.2025'
    final : '15 декабря 2025' ✅

  [КолДней]  →  (no correction applied)

  [КолДнейПрописью]  →  (no correction applied)

  [ДатаНачала]  →  (no corr

In [ ]:
# -- STEP 12: LOG ANALYSIS -- last 50 correction runs --------------------
# Reads tmp_logs/corrections.jsonl written by the pipeline after each run.

import sys, os
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from word_constructor.ai_correction import log_store

entries = log_store.read_all()
print(f"Total stored log entries: {len(entries)}")
if not entries:
    print("No log entries yet. Run cell [STEP 10] first, then re-run this cell.")
else:
    # ── Summary table ──────────────────────────────────────────────────────
    try:
        import pandas as pd
        rows = []
        for e in entries:
            timing = e.get("timing_ms") or {}
            tokens = e.get("tokens") or {}
            rows.append({
                "ts": (e.get("ts") or "")[:19],
                "log_key": (e.get("log_key") or "")[:28],
                "status": e.get("status", ""),
                "slots": e.get("slot_count", 0),
                "occ": e.get("occurrence_count", 0),
                "ai_sent": e.get("ai_sent_count", 0),
                "dedup": e.get("dedup_count", 0),
                "changed": e.get("changed_count", 0),
                "ai_ms": timing.get("ai_call", ""),
                "total_ms": timing.get("total", ""),
                "tokens_total": tokens.get("total_tokens", ""),
                "model": (e.get("model") or "")[-14:],
            })
        df = pd.DataFrame(rows)
        pd.set_option("display.max_colwidth", 30)
        pd.set_option("display.width", 120)
        print("
Last 10 runs:")
        display(df.tail(10))
    except ImportError:
        print("(install pandas for a table view)")
        for e in entries[-5:]:
            timing = e.get("timing_ms") or {}
            tokens = e.get("tokens") or {}
            print(f"  {(e.get('ts') or '')[:19]}  key={(e.get('log_key') or '')[:24]}  status={e.get('status')}  "
                  f"changed={e.get('changed_count',0)}/{e.get('occurrence_count',0)}  "
                  f"ai_ms={timing.get('ai_call','?')}ms  tokens={tokens.get('total_tokens','?')}")

    # ── Per-occurrence breakdown for the latest entry ──────────────────────
    latest = entries[-1]
    ts = (latest.get("ts") or "")[:19]
    key = latest.get("log_key") or ""
    print(f"
-- Latest run: {ts} | log_key={key} | status={latest.get('status')} --")
    for corr in latest.get("corrections", []):
        marker = "CHANGED" if corr.get("changed") else "      -"
        src = (corr.get("source") or "").ljust(13)
        ph = corr.get("placeholder", "")
        idx = corr.get("occurrence_index", 0)
        orig = corr.get("original", "")
        ai_raw = corr.get("ai_raw", "")
        final = corr.get("final", "")
        if corr.get("changed"):
            if ai_raw and ai_raw != orig and ai_raw != final:
                change_str = f"{orig!r} --(ai)--> {ai_raw!r} --(det)--> {final!r}"
            else:
                change_str = f"{orig!r} --> {final!r}"
        else:
            change_str = repr(orig)
        print(f"  {marker}  [{ph}]#{idx}  [{src}]  {change_str}")
    timing = latest.get("timing_ms") or {}
    tokens = latest.get("tokens") or {}
    print(f"
  Timing: ai_call={timing.get('ai_call','?')}ms  total={timing.get('total','?')}ms")
    print(f"  Tokens: prompt={tokens.get('prompt_tokens','?')}  completion={tokens.get('completion_tokens','?')}  total={tokens.get('total_tokens','?')}")
    print(f"  Log file: {log_store._log_path()}")
